# 原39项因子与 SOXL/现金 H5 区间方向：训练期选序审计

先在2011–2022 Qlib训练标签上报告全39项并固定前五，再只读2023–2026各年季度及f696全部819窗口。未来标签只作解释，不能变成决策规则。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json,gzip
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_original39_pairwise_factor_interval_audit_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
a=root/s['dataset_card'];f=root/s['f696_card'];b=root/s['account_card'];orig=root/s['original39_card']
for path,key in [(a/'summary.json','dataset_summary_sha256'),(f/'summary.json','f696_summary_sha256'),(b/'summary.json','account_summary_sha256'),(orig/'summary.json','original39_summary_sha256'),(root/s['regime_plan'],'regime_plan_sha256'),(root/s['provider']/'qlib_store_manifest.json','provider_manifest_sha256')]:assert sha(path)==s[key],key
sa=json.loads((a/'summary.json').read_text());sf=json.loads((f/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text());so=json.loads((orig/'summary.json').read_text())
assert sa['nas_readback']['verified'] and sa['nas_readback']['sha256']==s['dataset_NAS_sha256']
rb=json.loads((f/'readback_verification.json').read_text());assert rb['status']=='passed' and rb['all_artifact_hashes_match'] and rb['archive_sha256']==s['f696_NAS_sha256']
assert sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==s['account_NAS_sha256']
for source,exp,rid,art,key in [(a,'dataset_experiment_id','dataset_recorder_id','inputs/dataset','dataset_artifact_sha256'),(f,'f696_experiment_id','f696_recorder_id','inputs/frame','f696_frame_sha256'),(b,'account_experiment_id','account_recorder_id','analysis/rolling126','account_rolling_sha256')]:assert sha(source/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['dataset_experiment_id'],recorder_id=s['dataset_recorder_id']);dataset=rec.load_object('inputs/dataset')
tr_full=dataset.prepare('train',col_set=['feature','label','meta']);tr=tr_full.loc[tr_full[('meta','train_mature')].astype(bool)]
assert len(tr)==3014
features=list(tr['feature'].columns);assert len(features)==39 and features==so['input_audit']['original_features']
train_x=tr['feature'][features];train_y=tr[('label','SOXL_relative_H5')].astype(float)-tr[('label','cash_advantage_H5')].astype(float)
assert np.isfinite(train_y.to_numpy(float)).all()
def corr(x,y):
 z=pd.DataFrame({'x':x.to_numpy(float),'y':y.to_numpy(float)}).replace([np.inf,-np.inf],np.nan).dropna()
 return {'rows':len(z),'spearman':float(z.x.corr(z.y,method='spearman')) if len(z)>2 and z.x.nunique()>1 and z.y.nunique()>1 else None}
train_stats={name:corr(train_x[name],train_y) for name in features}
assert all(z['rows']>=2900 and z['spearman'] is not None for z in train_stats.values())
top5=sorted(features,key=lambda name:(-abs(train_stats[name]['spearman']),name))[:5]
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['f696_experiment_id'],recorder_id=s['f696_recorder_id']);frame=rec.load_object('inputs/frame').sort_values('trade_date').reset_index(drop=True)
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['account_experiment_id'],recorder_id=s['account_recorder_id']);rolling=rec.load_object('analysis/rolling126')
he_full=dataset.prepare('held',col_set=['feature','label','meta']);he=he_full.loc[he_full[('meta','held_mature')].astype(bool)]
assert len(he_full)==len(frame)==944 and len(he)==940 and len(rolling)==819
held_dates=pd.DatetimeIndex(he.index.get_level_values('datetime'));full_dates=pd.DatetimeIndex(he_full.index.get_level_values('datetime'))
assert full_dates.equals(pd.DatetimeIndex(frame.signal_date)) and held_dates.equals(full_dates[:940])
held_x=he['feature'][features];held_y=he[('label','SOXL_relative_H5')].astype(float)-he[('label','cash_advantage_H5')].astype(float)
assert np.isfinite(held_y.to_numpy(float)).all()
all_features=[]
for name in features:
 annual=[]
 for year in (2023,2024,2025,2026):
  mask=held_dates.year==year;annual.append({'year':year,'partial_endpoint':year in (2023,2026),**corr(held_x.loc[mask,name],held_y.loc[mask])})
 overall=corr(held_x[name],held_y)
 sign=np.sign(train_stats[name]['spearman'])
 all_features.append({'feature':name,'train':train_stats[name],'held':overall,'annual':annual,
                      'same_train_sign_complete_2024_2025':bool(all(np.sign(v['spearman'])==sign for v in annual if v['year'] in (2024,2025)))})
quarters=[]
for year in (2023,2024,2025,2026):
 for q in (1,2,3,4):
  mask=(held_dates.year==year)&(held_dates.quarter==q)
  if mask.any():quarters.append({'year':year,'quarter':q,'rows':int(mask.sum()),'selected':{name:corr(held_x.loc[mask,name],held_y.loc[mask]) for name in top5}})
assert len(quarters)==16
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))
close=D.features(['SOXX'],['$close'],start_time='2011-01-03',end_time='2026-10-09',freq='day')['$close'].droplevel('instrument').sort_index()
assert close.index.equals(calendar) and full_dates.isin(calendar).all()
windows=[]
for i in range(819):
 rw=rolling.iloc[i];gframe=frame.iloc[i:i+126];assert pd.Timestamp(gframe.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 signal=pd.Timestamp(gframe.signal_date.iloc[0]);pos=calendar.get_loc(signal);assert pos>=20
 r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 state='uptrend' if r5>=0 and r20>=0 else 'rebound' if r5>=0 else 'pullback' if r20>=0 else 'decline'
 end=min(i+126,940);gx=held_x.iloc[i:end];gy=held_y.iloc[i:end]
 assert 122<=len(gy)<=126
 row={'start_trade':str(pd.Timestamp(rw.first_trade).date()),'year':int(pd.Timestamp(rw.first_trade).year),'start_state':state,
      'outcome':'weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie','mature':len(gy),
      'selected':{name:corr(gx[name],gy) for name in top5}}
 windows.append(row)
w=pd.DataFrame(windows);assert w.outcome.value_counts().to_dict()=={'weak':564,'good':255}
cells=[]
for (state,year),g in w.groupby(['start_state','year'],sort=True):
 weak=g[g.outcome.eq('weak')];good=g[g.outcome.eq('good')]
 row={'start_state':state,'year':int(year),'weak_windows':len(weak),'good_windows':len(good),'matched':bool(len(weak) and len(good))}
 if row['matched']:
  row['weak_minus_good_mean_spearman']={name:float(weak.selected.map(lambda z:z[name]['spearman']).mean()-good.selected.map(lambda z:z[name]['spearman']).mean()) for name in top5}
 cells.append(row)
matched=[z for z in cells if z['matched']];assert len(matched)==12 and all(z['good_windows']==0 for z in cells if z['year']==2023)
weights=np.array([min(z['weak_windows'],z['good_windows']) for z in matched],float)
matched_diff={name:float(np.average([z['weak_minus_good_mean_spearman'][name] for z in matched],weights=weights)) for name in top5}
result={'status':'read_only_original39_train_ranked_pairwise_factor_interval_diagnostic_no_policy','plan_sha256':sha(plan_path),
        'source_recorder_ids':{'dataset':s['dataset_recorder_id'],'f696':s['f696_recorder_id'],'account':s['account_recorder_id']},
        'train_mature_rows':len(tr),'held_mature_rows':len(he),'held_full_source_rows':len(frame),'unknown_held_labels':4,'original_feature_count':len(features),
        'train_selected_top5':top5,'all39':all_features,'held_quarter_selected':quarters,'rolling126':windows,'rolling126_windows':len(w),
        'matched_state_year_cells':len(matched),'matched_cells':cells,'matched_weak_minus_good_spearman':matched_diff,
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['39 training correlations are a multiple-comparison descriptive screen, not a new fitted factor or causal importance','H5 gross-price future margins are unavailable at Stage1 decision time','Repeated held market history and overlapping weak/good windows are not independent validation or a policy threshold']}
rolling_payload=json.dumps(result.pop('rolling126'),ensure_ascii=False,separators=(',',':')).encode()
rolling_gzip=gzip.compress(rolling_payload,mtime=0)
rolling_path=card/'rolling126.json.gz';rolling_path.write_bytes(rolling_gzip)
result['rolling126_gzip_path']=rolling_path.name;result['rolling126_gzip_sha256']=hashlib.sha256(rolling_gzip).hexdigest()
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'top5':[(name,train_stats[name]['spearman']) for name in top5],
       'held_years':{name:[(z['year'],z['spearman']) for z in next(x for x in all_features if x['feature']==name)['annual']] for name in top5},
       'matched':matched_diff})


{'top5': [('cross_vol_breadth_high', 0.07268026895647743), ('soxl_ret3', -0.07037970952611793), ('soxx_trend60', -0.06745336878779414), ('soxx_ret3', -0.06604591708552918), ('soxx_drawdown60', -0.06427395372801899)], 'held_years': {'cross_vol_breadth_high': [(2023, 0.18942121467899684), (2024, 0.07262920361129652), (2025, -0.053074677743828985), (2026, -0.0854914677679572)], 'soxl_ret3': [(2023, -0.04902568041088658), (2024, -0.09858926799858725), (2025, -0.07152613641818267), (2026, -0.043636797078599326)], 'soxx_trend60': [(2023, -0.1036374981999712), (2024, -0.12117612741205327), (2025, -0.05167378678058849), (2026, 0.005331339499049135)], 'soxx_ret3': [(2023, -0.04937589401430422), (2024, -0.10144477613731034), (2025, -0.07865559448951182), (2026, -0.04632233293059325)], 'soxx_drawdown60': [(2023, -0.09228742263891064), (2024, -0.10561889712363574), (2025, -0.058638961738709805), (2026, 0.039083684856637144)]}, 'matched': {'cross_vol_breadth_high': -0.030177984869443196, 'soxl_ret3

## 中文结论

# 原39项对SOXL/现金五日边际的训练相关很弱，短收益负向较稳定

严格先在原Qlib DatasetH的2011–2022 **3014** 个成熟标签上计算全部39项决策前因子对 `SOXL_relative_H5−cash_advantage_H5` 毛价格边际的完整配对Spearman，按训练绝对值与名称固定前五项后，才只读2023–2026的 **940** 个成熟复用标签（另4个保持未知）。精确f696账户的944日、16季度及819个重叠126日窗口全部纳入；窗口逐项结果压缩保存在`rolling126.json.gz`，摘要哈希见`result.json`。0新市场GET、模型、账户或bootstrap。

训练前五及相关系数：`cross_vol_breadth_high` **+0.07268**、`soxl_ret3` **−0.07038**、`soxx_trend60` **−0.06745**、`soxx_ret3` **−0.06605**、`soxx_drawdown60` **−0.06427**。幅度都很小。跨资产波动广度在完整2025年变成 **−0.05307**，部分2026 **−0.08549**；而`soxl_ret3`与`soxx_ret3`在2023、完整2024/2025及部分2026保持负号，评价全段分别 **−0.06114/−0.06389**。39项中仅19项与训练符号在完整2024/2025两年都一致；符号一致不等于费用后净收益。

按窗口起点前SOXX 5/20日走势状态与首交易年配对，12个格子同时有好坏窗口；2023四格全部没有同格好窗口。弱减好窗口内目标秩相关的加权差，`soxl_ret3` **−0.07794**、`soxx_ret3` **−0.08191**；这是未来收益定义且高度重叠的窗口归因，不是独立统计或可交易阈值。所有39项全年、五项16季度与819窗、配对格子均可从sidecar复核。

决定 `descriptive_original39_pairwise_train_rank_small_effect_no_policy`。原Stage1已接收这些短收益坐标，不能把一个约0.07的训练期单列相关包装成新信息；此前原39项直接二分类固定版本在2025后失效。若提出短期反转的**不同经济机制**，须先证明对f696配置排序及相应风险容量的增量、来源时钟和合法动作支持，再冻完整0bp Tiered自身状态账户；不能从这张复用历史表挑年份、翻号或马上推广。SOXX目标未达成。
